In [3]:
#Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [7]:
#Helper Functions
def add_user_message(messages, text):
    user_message = { "role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, ore Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representign task that requires Python, JSON or a Regex to complete. 

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex",
            "solution_criteria": "Must include runtime, memory size, timeout, and basic structure for AWS Lambda configuration"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python functin, a single JSON object, or a single regex.
    * Focus on tasks that do not require writing much code.

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open("dataset1.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [26]:
def run_prompt(test_case):
    """Merge the prompt and test case input, then return the results"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON or a plain Regex
    * Do not add any comments or commentary or explaination
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [28]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10 
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [29]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [30]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [ ]:
with open("dataset1.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.833333333333333


In [33]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:GetObjectVersion\",\n        \"s3:ListBucket\",\n        \"s3:ListBucketVersions\",\n        \"s3:GetBucketLocation\",\n        \"s3:GetBucketVersioning\",\n        \"s3:ListAllMyBuckets\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::*\",\n        \"arn:aws:s3:::*/*\"\n      ]\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create a JSON policy document that allows read-only access to all S3 buckets for a specific IAM user",
      "format": "json"
    },
    "score": 8.5,
    "reasoning": "The policy correctly implements read-only S3 access with appropriate actions and resource ARNs. However, it fails to address the key requirement of limiting access to a specific IAM user, instead creating a general policy that could be applied to any principal."
  },
  {
    "output": "\ndef extract_region_from